# 01 — Inventory & Format Contract  *(Block A)*

**Run this first. Nothing else matters until it passes.**

KLA scores images *exactly as saved* by our pipeline and performs no clipping or
renormalisation (problem statement §4A). If our output format does not match the
ground-truth format, the score is capped by an I/O bug rather than by the model.

Four questions, in order of consequence:

1. What format/dtype is the GT, and can we reproduce it bit-exactly?
2. Are pairs complete, grayscale, and exactly 2x?
3. How far outside `[0,1]` does NoisyLR actually go?
4. Is the intensity range stable enough for fixed global normalisation?

In [ ]:
# --- Kaggle setup ---------------------------------------------------------
# !git clone -q https://github.com/<org>/kla-restoration.git
# %cd kla-restoration
# !pip install -q -r requirements.txt

import sys, os
REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, REPO)

%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

from src.io_utils import detect_format, round_trip_check, load_image, list_images, pair_by_stem
from src.transforms import save_stats

DATA = Path("/kaggle/input/kla-dataset")   # <-- point this at the real dataset
GT_DIR, LR_DIR = DATA / "GT", DATA / "NoisyLR"

# Sanity: what is actually in there?
print("exists:", DATA.exists())
for p in sorted(DATA.glob("*"))[:20]:
    print("  ", p.name, "(dir)" if p.is_dir() else "")

In [ ]:
pairs = pair_by_stem(GT_DIR, LR_DIR)
n_gt, n_lr = len(list_images(GT_DIR)), len(list_images(LR_DIR))
print(f"GT files {n_gt} | LR files {n_lr} | matched pairs {len(pairs)}")
if len(pairs) < min(n_gt, n_lr):
    print(f"!! {min(n_gt,n_lr)-len(pairs)} file(s) failed to pair by stem - check naming")
pairs[:3]

## 1. Format contract — the highest-risk check

`round_trip_check` loads a GT file, saves it back through our own writer, reloads
it, and compares. **If this fails, stop and fix `src/io_utils.py` before anything
else.**

In [ ]:
gt_fmt = detect_format(pairs[0][0])
lr_fmt = detect_format(pairs[0][1])
rt = round_trip_check(pairs[0][0])

print("GT format:", gt_fmt)
print("LR format:", lr_fmt)
print(f"round-trip max_abs_error = {rt['max_abs_error']:.3e}  (allowed {rt['allowed']:.3e})")
print("PASS" if rt["passed"] else "FAIL  <-- STOP, fix io_utils before training")

## 2–4. Per-pair statistics

In [ ]:
rows = []
for gt_p, lr_p in pairs:
    g, l = load_image(gt_p), load_image(lr_p)
    rows.append(dict(
        stem=gt_p.stem, gt_h=g.shape[0], gt_w=g.shape[1], lr_h=l.shape[0], lr_w=l.shape[1],
        scale_h=g.shape[0]/l.shape[0], scale_w=g.shape[1]/l.shape[1],
        gt_min=g.min(), gt_max=g.max(), gt_mean=g.mean(), gt_std=g.std(),
        lr_min=l.min(), lr_max=l.max(), lr_mean=l.mean(), lr_std=l.std(),
        frac_above1=float((l > 1).mean()), frac_below0=float((l < 0).mean()),
    ))
df = pd.DataFrame(rows)
Path("../artifacts").mkdir(exist_ok=True)
df.to_csv("../artifacts/inventory.csv", index=False)
df.head()

In [ ]:
print("=== SHAPES ===")
print(df.groupby(["gt_h","gt_w","lr_h","lr_w"]).size().rename("count"))
bad = df[(df.scale_h != 2) | (df.scale_w != 2)]
print(f"\npairs not exactly 2x: {len(bad)}")
if len(bad): display(bad)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

sample = np.random.default_rng(0).choice(len(pairs), size=min(30, len(pairs)), replace=False)
gt_vals = np.concatenate([load_image(pairs[i][0]).ravel()[::17] for i in sample])
lr_vals = np.concatenate([load_image(pairs[i][1]).ravel()[::17] for i in sample])

ax[0].hist(gt_vals, bins=120, alpha=.6, label="GT", density=True)
ax[0].hist(lr_vals, bins=120, alpha=.6, label="NoisyLR", density=True)
ax[0].axvline(0, ls="--", c="k", lw=.8); ax[0].axvline(1, ls="--", c="k", lw=.8)
ax[0].set_title("intensity distribution"); ax[0].legend(); ax[0].set_yscale("log")

ax[1].hist(df.frac_above1*100, bins=30)
ax[1].set_title("% of LR pixels > 1, per image"); ax[1].set_xlabel("%")

ax[2].hist(df.gt_mean, bins=30)
ax[2].set_title("per-image GT mean"); ax[2].set_xlabel("mean intensity")
plt.tight_layout(); plt.show()

print(f"GT range  [{df.gt_min.min():.4f}, {df.gt_max.max():.4f}]   (spec: normalised to [0,1])")
print(f"LR range  [{df.lr_min.min():.4f}, {df.lr_max.max():.4f}]")
print(f"LR > 1 : {df.frac_above1.mean()*100:.3f}% of pixels (worst image {df.frac_above1.max()*100:.3f}%)")
print(f"LR < 0 : {df.frac_below0.mean()*100:.3f}%")

### Normalisation decision

Low CV of per-image means → fixed global scaling. That is deterministic, costs
nothing at inference and, crucially, does not depend on statistics of an unseen
image — which is exactly where per-image normalisation breaks on out-of-distribution data.

In [ ]:
cv = df.gt_mean.std() / df.gt_mean.mean()
print(f"CV of per-image GT means = {cv:.4f}")
print("-> fixed global scaling, scale_constant = 1.0" if cv < 0.15
      else "-> range drifts; consider dataset-level (NOT per-image) z-score")

## Visual check

Always look at the images, not just the numbers. The spec warns against blurring
to remove noise — you need to know what the structure actually looks like.

In [ ]:
k = 3
fig, axes = plt.subplots(k, 3, figsize=(13, 4.3*k))
for r, i in enumerate(np.random.default_rng(1).choice(len(pairs), k, replace=False)):
    g, l = load_image(pairs[i][0]), load_image(pairs[i][1])
    axes[r,0].imshow(g, cmap="gray", vmin=0, vmax=1); axes[r,0].set_title(f"GT {g.shape}")
    axes[r,1].imshow(l, cmap="gray", vmin=0, vmax=1); axes[r,1].set_title(f"NoisyLR {l.shape}")
    axes[r,2].imshow(g[:96,:96], cmap="gray", vmin=0, vmax=1); axes[r,2].set_title("GT detail 96px")
    for a in axes[r]: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
stats = {
    "scale_constant": 1.0,
    "log_transform": False,          # 02_degradation.ipynb may flip this
    "log_eps": 0.01,
    "gt_valid_range": [0.0, 1.0],
    "gt_format": gt_fmt.to_dict(),   # <- the output contract for the whole team
    "roundtrip_passed": bool(rt["passed"]),
    "n_pairs": len(pairs),
    "cv_image_means": float(cv),
    "frac_lr_above_1": float(df.frac_above1.mean()),
}
save_stats(stats, "../artifacts/stats.json")
print("wrote ../artifacts/stats.json")
stats

---
### Post to the team

> **Output format contract:** `<gt_fmt>` — outputs must be saved in exactly this
> format/dtype. Round-trip verified. `artifacts/stats.json` is committed; import
> `normalize` / `denormalize` from `src.transforms` and do not hardcode constants
> anywhere else.
>
> **`val_ood` is the primary metric**, `val_id` is a sanity check.